# backward-func-lookup — ex2: dispatch through BackwardFuncLookup for a 2-op reverse pass

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backward-func-lookup`. Running the final beacon cell reports progress against the `Backprop: BackwardFuncLookup` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: BackwardFuncLookup` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-func-lookup`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-func-lookup"
DD_SUBTOPIC = "Backprop: BackwardFuncLookup"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## BackwardFuncLookup in a tiny reverse pass — quick refresher

The lookup is keyed by `(forward_fn, arg_position)` and dispatched at the moment the reverse pass walks a node's parents:

```python
for argnum, parent in node.recipe.parents.items():
    back_fn = BACK_FUNCS.get_back_func(node.recipe.func, argnum)
    grad_in = back_fn(grad_out, node.array, *node.recipe.args)
    grads[parent] = grads.get(parent, 0) + grad_in
```

Two dispatch invariants that ex1's basic register-and-get tests do NOT exercise:
- **Per-argnum independence.** `(multiply, 0)` and `(multiply, 1)` route to DIFFERENT back fns even though the forward is symmetric. The dispatcher does not know about symmetry — it always asks for `(func, argnum)`.
- **Composition.** The lookup is called ONCE PER PARENT EDGE in the reverse pass. For an op with `K` tensor parents, dispatch happens `K` times — once per `(func, argnum)` pair.

### Exercise 2 — dispatch through BackwardFuncLookup for a 2-op reverse pass

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply BackwardFuncLookup to dispatch the correct back fn for each (func, argnum) pair across two composed ops in a mini reverse pass.
> Keywords: dispatch, reverse-pass, argnum, compose
> ```

**KCs targeted:** `backward-func-lookup`, `register-back-fn-after-wrap`

Implement `mini_reverse_pass(end_node, end_grad, BACK_FUNCS)` — a minimal reverse pass that consumes a pre-sorted graph and dispatches back fns via the lookup.

Inputs:
- `end_node`: a `MiniTensor` at the tip of a compute graph.
- `end_grad`: a raw `torch.Tensor` (already resolved, same shape as `end_node.array`).
- `BACK_FUNCS`: a populated `BackwardFuncLookup`.

Output: a dict `grads: dict[int, torch.Tensor]` keyed by `id(leaf)`, value = accumulated gradient for that leaf.

Algorithm:

```python
node_grads = {id(end_node): end_grad}
for node in sorted_graph:                 # end node first
    if node.recipe is None:               # leaf — record and skip
        continue
    grad_out = node_grads[id(node)]
    for argnum, parent in node.recipe.parents.items():
        back_fn = BACK_FUNCS.get_back_func(node.recipe.func, argnum)
        grad_in = back_fn(grad_out, node.array, *node.recipe.args)
        prev = node_grads.get(id(parent))
        node_grads[id(parent)] = grad_in if prev is None else prev + grad_in
```

We've provided `sorted_computational_graph` (drops in via a tiny reverse-topological-walk) and `BackwardFuncLookup` for you. The two ops you'll be exercising are `t.log` and `t.multiply`.

**The point of this drill.** It's NOT about implementing topo-sort or the math — both are upstream prerequisites you already have. It's about the LOOKUP CALL: `BACK_FUNCS.get_back_func(func, argnum)` runs ONCE PER PARENT EDGE in the reverse pass. For an op with 2 tensor parents, the lookup is invoked TWICE per node — once with each argnum.

In [ ]:
def mini_reverse_pass(end_node, end_grad, BACK_FUNCS):
    node_grads = {id(end_node): end_grad}
    id_to_node = {id(end_node): end_node}
    for node in sorted_computational_graph(end_node):
        id_to_node[id(node)] = node
        if node.recipe is None:
            continue
        grad_out = node_grads[id(node)]
        for argnum, parent in node.recipe.parents.items():
            back_fn = BACK_FUNCS.get_back_func(node.recipe.func, argnum)
            grad_in = back_fn(grad_out, node.array, *node.recipe.args)
            prev = node_grads.get(id(parent))
            node_grads[id(parent)] = grad_in if prev is None else prev + grad_in
    return node_grads


<details><summary>Solution</summary>

```python
def mini_reverse_pass(end_node, end_grad, BACK_FUNCS):
    node_grads = {id(end_node): end_grad}
    id_to_node = {id(end_node): end_node}
    for node in sorted_computational_graph(end_node):
        id_to_node[id(node)] = node
        if node.recipe is None:
            continue
        grad_out = node_grads[id(node)]
        for argnum, parent in node.recipe.parents.items():
            back_fn = BACK_FUNCS.get_back_func(node.recipe.func, argnum)
            grad_in = back_fn(grad_out, node.array, *node.recipe.args)
            prev = node_grads.get(id(parent))
            node_grads[id(parent)] = grad_in if prev is None else prev + grad_in
    return node_grads
```

**Why dispatch matters even on this tiny graph.** The two `(t.multiply, argnum)` registrations point to DIFFERENT back fns even though the math is symmetric. The reverse pass would crash with a `KeyError` or produce wrong grads if the lookup keyed only on `func` and ignored `argnum`.

**Why `id(parent)` keys.** Two MiniTensors with equal `.array` tensors are still SEPARATE leaves — they accumulate their own grads. Identity keys avoid accidental merging. The same pattern shows up in real frameworks: PyTorch's autograd graph keys by tensor identity, not value.

**Re-register exercises the dispatch path.** Hardcoding `log_back` and `multiply_back0` inside the loop would pass the first half of the test. The 'doubled back fn' test exposes that — only an implementation that goes through `BACK_FUNCS.get_back_func(...)` every call sees the re-registration.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()